# 8. Debt Recovery Strategy

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

trained_models = joblib.load("../data/05_trained_models.pkl")
processed_data = joblib.load("../data/03_processed_data.pkl")

random_forest_model = trained_models["random_forest_model"]
X_test = processed_data["X_test"]
y_test = processed_data["y_test"]

# Check that the saved model and test data use the same features.
expected_features = getattr(random_forest_model, "n_features_in_", None)
if expected_features is not None and X_test.shape[1] != expected_features:
    raise ValueError(
        f"Feature mismatch: model expects {expected_features} features, "
        f"but X_test contains {X_test.shape[1]}."
    )

if len(X_test) != len(y_test):
    raise ValueError(
        f"Row mismatch: X_test has {len(X_test)} rows, "
        f"but y_test has {len(y_test)} rows."
    )

print("Model and test data loaded successfully!")
print("Test data shape:", X_test.shape)
print("Random Forest model:", type(random_forest_model).__name__)


Model and test data loaded successfully!
Test data shape: (4940, 21)
Random Forest model: RandomForestClassifier


## Debt Recovery Strategy

The machine learning and explainability results can be used to segment
customers and prioritize recovery efforts.

Customers with higher recovery likelihood can be handled using
standardized and lower-cost communication methods, while customers with
lower recovery likelihood can receive more proactive and personalized
follow-up.

The strategy is based on model predictions and the factors identified
through feature importance and SHAP analysis.

In [2]:
# Predict the probability that each loan will be recovered (class 1).
recovery_probability = random_forest_model.predict_proba(X_test)[:, 1]

# Convert the target to a one-dimensional array for reliable DataFrame creation.
actual_values = np.asarray(y_test).reshape(-1)

recovery_segments = pd.DataFrame({
    "Actual": actual_values,
    "Recovery_Probability": recovery_probability
})

# Group accounts by predicted recovery probability.
recovery_segments["Segment"] = pd.cut(
    recovery_segments["Recovery_Probability"],
    bins=[0.0, 0.40, 0.70, 1.0],
    labels=["Low Recovery", "Medium Recovery", "High Recovery"],
    include_lowest=True
)

print("Number of test accounts in each segment:")
display(recovery_segments["Segment"].value_counts(sort=False).rename("Accounts").to_frame())

display(recovery_segments.head())


Number of test accounts in each segment:


,Accounts
Segment,
Low Recovery,1188
Medium Recovery,970
High Recovery,2782


,Actual,Recovery_Probability,Segment
0,1,0.847659,High Recovery
1,0,0.360892,Low Recovery
2,1,0.985294,High Recovery
3,0,0.280268,Low Recovery
4,0,0.720485,High Recovery


In [3]:
segment_summary = (
    recovery_segments
    .groupby("Segment", observed=False)
    .agg(
        Accounts=("Actual", "size"),
        Actual_Recovery_Rate=("Actual", "mean"),
        Average_Predicted_Probability=("Recovery_Probability", "mean")
    )
)

segment_summary["Actual_Recovery_Rate"] = (
    segment_summary["Actual_Recovery_Rate"] * 100
).round(2)

segment_summary["Average_Predicted_Probability"] = (
    segment_summary["Average_Predicted_Probability"] * 100
).round(2)

print("Recovery summary by segment:")
display(segment_summary)


Recovery summary by segment:


,Accounts,Actual_Recovery_Rate,Average_Predicted_Probability
Segment,,,
Low Recovery,1188,13.22,18.49
Medium Recovery,970,52.27,56.46
High Recovery,2782,91.55,88.46


## Recommended Debt Recovery Strategy

Based on the model's predicted recovery probability and SHAP insights,
customers can be prioritized into three recovery segments.

### Low Recovery
Customers with predicted recovery probability below 40%.

**Recommended approach:**
- High-priority follow-up
- Personalized communication
- Review previous defaults and payment delays
- Consider suitable repayment assistance where appropriate

### Medium Recovery
Customers with predicted recovery probability between 40% and 70%.

**Recommended approach:**
- Regular follow-up
- Payment reminders
- Monitor payment behavior
- Encourage timely repayment

### High Recovery
Customers with predicted recovery probability above 70%.

**Recommended approach:**
- Standard reminders
- Lower-cost communication
- Automated follow-up
- Focus intensive recovery resources elsewhere

In [4]:
output_path = Path("../data/08_recovery_segments.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

recovery_segments.to_csv(output_path, index=False)
print(f"Recovery segments saved successfully to: {output_path.resolve()}")


Recovery segments saved successfully to: C:\Users\manth\Documents\HCL Final Project\data\08_recovery_segments.csv


## Business Considerations

The loan recovery prediction model can support debt recovery teams by prioritizing accounts according to their predicted recovery probability.

Instead of allocating the same level of recovery effort to every account, the model can help organize accounts into different priority levels.

### Low Recovery Probability

These accounts may require detailed review, personalized communication, and suitable repayment assistance.

### Medium Recovery Probability

These accounts can receive regular follow-ups, payment reminders, and continued monitoring.

### High Recovery Probability

These accounts can generally be handled through standard or automated follow-up processes, allowing recovery teams to focus intensive resources on higher-priority cases.

The model is intended to support decision-making and resource allocation, rather than completely replace human judgment.

## Ethical Considerations

Loan recovery predictions can affect how borrowers are prioritized for collection activities. Therefore, the model should be used responsibly.

### Fairness

The model should be evaluated to ensure that predictions do not unfairly disadvantage borrowers based on sensitive or protected characteristics.

### Transparency

Explainable AI techniques such as SHAP are used to understand which features influence model predictions.

### Privacy

Borrower information should be handled securely, and only the data required for the intended business purpose should be used.

### Human Oversight

Model predictions should support collection teams rather than make completely automated decisions about borrowers.

### Responsible Use

A low recovery probability should not automatically imply that a borrower is unwilling to repay. The prediction represents a statistical estimate based on historical data and should be considered together with relevant business and borrower context.

## Overall Project Conclusion

The project developed a machine learning-based loan recovery prediction system using historical borrower and loan information.

Logistic Regression, Decision Tree, and Random Forest classification models were evaluated using Recall, ROC-AUC, accuracy, and confusion matrix analysis.

Random Forest was selected because it achieved the highest recall among the evaluated models in the current comparison. Logistic Regression achieved the highest ROC-AUC and accuracy, so the final model choice reflects the project's priority of identifying loans that may be recovered, while recognizing that model selection involves trade-offs.

SHAP analysis provided explainable insights into the factors influencing recovery predictions.

Predicted recovery probabilities were used to divide test accounts into Low, Medium, and High Recovery segments. These segments can help organize follow-up strategies, but they should support—not replace—human judgment and fair treatment of borrowers.